# Single-path backtest

Compare the **equity book** vs **sticky-alpha options overlay** on one simulated market path.
For strategy validity across many paths, use the UI Monte Carlo run (or `backend.monte_carlo`).

This notebook replaces the old split `notebook-logging` / `notebook-charts` pair.


In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from backend.models import (
    SimulationParams,
    UniverseParams,
    FinancingParams,
    StrategyParams,
    OptionsParams,
)
from backend.simulator import run_simulation

SEED = 42
WRITE_DETAIL_LOG = False  # set True to dump period-by-period log via run_strategy


## Parameters


In [ ]:
params = SimulationParams()  # defaults from backend.models (lift-friendly)
params


## Run


In [ ]:
sim = run_simulation(params, seed=SEED)
base, overlay, bench = sim.base, sim.with_options, sim.benchmark

n = min(len(base.portfolio_returns), len(overlay.portfolio_returns), len(bench.returns))
months = np.arange(1, n + 1)

def _frame(result, *, with_options: bool) -> pd.DataFrame:
    data = {
        "month": months,
        "ret": result.portfolio_returns[:n],
        "turnover": result.turnovers[:n],
        "financing": result.financing_costs[:n],
    }
    if with_options:
        data["options_income"] = result.options_income[:n]
    df = pd.DataFrame(data)
    df["cum"] = (1.0 + df["ret"] / 100.0).cumprod() - 1.0
    peak = (1.0 + df["cum"]).cummax()
    df["drawdown"] = (1.0 + df["cum"]) / peak - 1.0
    return df

df_base = _frame(base, with_options=False)
df_ovl = _frame(overlay, with_options=True)
df_bench = pd.DataFrame({"month": months, "ret": bench.returns[:n]})
df_bench["cum"] = (1.0 + df_bench["ret"] / 100.0).cumprod() - 1.0

df_ovl["lift_cum"] = (df_ovl["cum"] - df_base["cum"]) * 100.0  # pp vs base


## Summary stats


In [ ]:
def _max_dd(cum: pd.Series) -> float:
    wealth = 1.0 + cum
    return float((wealth / wealth.cummax() - 1.0).min() * 100.0)

summary = pd.DataFrame(
    {
        "Base (no options)": {
            "Ann. return %": base.annualized_return,
            "Sharpe": base.sharpe_ratio,
            "Max DD %": _max_dd(df_base["cum"]),
            "Avg turnover %": base.avg_turnover,
            "Avg financing % ann.": base.avg_financing_cost,
            "Avg options income % ann.": base.avg_options_income,
            "Alpha % ann.": sim.alpha_base,
            "IR": sim.information_ratio_base,
        },
        "Overlay": {
            "Ann. return %": overlay.annualized_return,
            "Sharpe": overlay.sharpe_ratio,
            "Max DD %": _max_dd(df_ovl["cum"]),
            "Avg turnover %": overlay.avg_turnover,
            "Avg financing % ann.": overlay.avg_financing_cost,
            "Avg options income % ann.": overlay.avg_options_income,
            "Alpha % ann.": sim.alpha_with_options,
            "IR": sim.information_ratio_with_options,
        },
        "Benchmark": {
            "Ann. return %": bench.annualized_return,
            "Sharpe": bench.sharpe_ratio,
            "Max DD %": _max_dd(df_bench["cum"]),
            "Avg turnover %": np.nan,
            "Avg financing % ann.": np.nan,
            "Avg options income % ann.": np.nan,
            "Alpha % ann.": 0.0,
            "IR": np.nan,
        },
    }
).round(3)

print(f"Options lift (ann. return overlay − base): {sim.options_lift:.3f}%")
print(f"Terminal cumulative lift: {df_ovl['lift_cum'].iloc[-1]:.3f} pp")
summary


## Charts


In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=df_base["month"], y=df_base["cum"] * 100, name="Base", line=dict(color="#2563eb")))
fig.add_trace(go.Scatter(x=df_ovl["month"], y=df_ovl["cum"] * 100, name="Overlay", line=dict(color="#16a34a")))
fig.add_trace(go.Scatter(x=df_bench["month"], y=df_bench["cum"] * 100, name="Benchmark", line=dict(color="#6b7280", dash="dash")))
fig.update_layout(
    title="Cumulative returns",
    xaxis_title="Month",
    yaxis_title="Return (%)",
    hovermode="x unified",
    template="plotly_white",
)
fig.show()


In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=df_base["month"], y=df_base["drawdown"] * 100, name="Base DD", line=dict(color="#2563eb")))
fig.add_trace(go.Scatter(x=df_ovl["month"], y=df_ovl["drawdown"] * 100, name="Overlay DD", line=dict(color="#16a34a")))
fig.update_layout(
    title="Drawdowns",
    xaxis_title="Month",
    yaxis_title="Drawdown (%)",
    hovermode="x unified",
    template="plotly_white",
)
fig.show()


In [ ]:
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    subplot_titles=("Cumulative options lift (pp vs base)", "Monthly options income (% of NAV)"))
fig.add_trace(go.Scatter(x=df_ovl["month"], y=df_ovl["lift_cum"], name="Lift", line=dict(color="#ca8a04")), row=1, col=1)
fig.add_trace(go.Bar(x=df_ovl["month"], y=df_ovl["options_income"], name="Options income", marker_color="#f59e0b"), row=2, col=1)
fig.update_layout(title="Options overlay P&L", hovermode="x unified", template="plotly_white", showlegend=False)
fig.update_yaxes(title_text="pp", row=1, col=1)
fig.update_yaxes(title_text="% NAV", row=2, col=1)
fig.update_xaxes(title_text="Month", row=2, col=1)
fig.show()


In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=df_base["month"], y=df_base["turnover"], name="Base turnover", line=dict(color="#2563eb")))
fig.add_trace(go.Scatter(x=df_ovl["month"], y=df_ovl["turnover"], name="Overlay turnover", line=dict(color="#16a34a")))
fig.update_layout(
    title="Monthly turnover (Σ|Δw| × 100)",
    xaxis_title="Month",
    yaxis_title="Turnover (%)",
    hovermode="x unified",
    template="plotly_white",
)
fig.show()

corr = np.corrcoef(df_ovl["turnover"], df_ovl["options_income"])[0, 1]
print(f"Corr(turnover, options income) on overlay path: {corr:.3f}")


## Optional detail log

Writes a period-by-period text log for the **overlay** book (same market seed). Useful for debugging a single month; skip for normal analysis.


In [ ]:
if WRITE_DETAIL_LOG:
    from backend.market import generate_market_data
    from backend.strategy import run_strategy
    from backend.optimizer import PortfolioOptimizer

    sp = params.strategy
    n_periods = sp.lookback + sp.strategy_length
    rng = np.random.default_rng(SEED)
    market = generate_market_data(
        n_periods, params.universe.n_assets,
        mean_return=params.universe.mean_return,
        volatility=params.universe.volatility,
        ic=params.universe.ic,
        factor_autocorr=params.universe.factor_autocorr,
        market_vol=params.universe.market_vol,
        market_autocorr=params.universe.market_autocorr,
        style_vol=params.universe.style_vol,
        style_autocorr=params.universe.style_autocorr,
        avg_beta=params.universe.avg_beta,
        beta_dispersion=params.universe.beta_dispersion,
        student_t_df=params.universe.student_t_df,
        stoch_vol_persistence=params.universe.stoch_vol_persistence,
        stoch_vol_of_vol=params.universe.stoch_vol_of_vol,
        rng=rng,
    )
    strategy_params = {
        "lookback": sp.lookback,
        "strategy_length": sp.strategy_length,
        "risk_aversion": sp.risk_aversion,
        "long_weight": sp.long_weight,
        "short_weight": sp.short_weight,
        "max_long_weight": sp.max_long_weight,
        "max_short_weight": sp.max_short_weight,
        "transaction_cost_bps": sp.transaction_cost_bps,
        "market_impact_coef": sp.market_impact_coef,
        "hard_turnover_limit": sp.hard_turnover_limit,
        "weight_threshold": sp.weight_threshold,
        "signal_ic": sp.signal_ic,
        "alpha_method": sp.alpha_method,
        "cov_method": sp.cov_method,
        "cov_halflife": sp.cov_halflife,
    }
    financing_params = params.financing.model_dump()
    options_params = {**params.options.model_dump(), "enabled": True}
    log_path = "detailed_strategy_log.txt"
    run_strategy(
        market.factor_scores, market.simple_returns, market.prices,
        strategy_params, financing_params, options_params,
        verbose=False, log_file=log_path,
    )
    print(f"Wrote {log_path}")
else:
    print("Skipped detail log (set WRITE_DETAIL_LOG = True to enable).")
